# 15: Runtime (E6, RQ6, gate G5)

Design decisions
- Production runtime: the seconds per distinct grid cell stored with every candidate cache of the main results, for the explained patients of calibration split 2 and of the test split (notebooks 08, 09 and 09d for XGBoost; notebook 09b for LightGBM and random forest), with the share of explained patients whose cell had already been computed (cell cache) and the share of soundness checks answered by the memo. These were measured in the earlier Colab sessions.
- Timing study in this session: the frozen candidate generator (M = 10, feature-order seed 42, search budget 200,000 nodes) with clocks on its interval pre-check, exact search, deletion and inflation steps; answers are unchanged. Each model explains up to 100 distinct cells, taken from the explained test patients in a seeded random order, within 240 seconds; a cell still running after 120 seconds is stopped and counted as censored at 120 seconds. The order does not depend on how hard a cell is, so a model stopped by the time limit still gives a fair sample; when half the cells are censored, the median is a lower bound and is marked so.
- Scaling on BRFSS Heart disease and Cardio-70k with 25, 50, 100 and 200 trees, in two series. Frozen settings (gamma 2, minimum child weight 50), depth 3 to 6: these settings stop trees from splitting after about 30 trees, so the effective size is reported next to the requested size (trees with at least one split, internal nodes, depth reached, distinct thresholds); the 100-tree, depth-4 model must reproduce the frozen model exactly. No regularisation (gamma 0, minimum child weight 1, other settings frozen), depth 3, 4 and 6: every requested tree can split, so this series shows the cost of fully grown ensembles. Each model is trained on the training portion with the frozen Youden rule, checked against the native library on the training portion, and its out-of-fold AUROC on the training portion is reported.
- CDC Diabetes: the frozen model is timed the same way for the split by step.
- RQ6 criterion: median of 5 seconds or less per distinct cell for 100 trees or fewer and depth 6 or less. Gate G5: median above 30 seconds per cell for a frozen model.
- Per-model timing rows are saved on Drive, so an interrupted run resumes; the hardware of each session is recorded.

In [ ]:
# === SESSION BOOTSTRAP (run first) ===
import subprocess, sys, warnings


def quiet_warnings():
    # imodels switches on every DeprecationWarning; Colab's message code then warns on every output line
    warnings.filterwarnings('ignore', category=DeprecationWarning, module='jupyter_client')
    warnings.filterwarnings('ignore', message=r'datetime\.datetime\.utcnow')


quiet_warnings()
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'z3-solver'], check=True)
from google.colab import drive
drive.mount('/content/drive')
import os, shutil
DRIVE_ROOT = '/content/drive/MyDrive/SCAX_Research'
REPO_DIR = f'{DRIVE_ROOT}/scax-research'
for name in ('.gitconfig', '.git-credentials'):
    if os.path.exists(f'{DRIVE_ROOT}/{name}'):
        shutil.copy(f'{DRIVE_ROOT}/{name}', os.path.expanduser(f'~/{name}'))
os.chdir(REPO_DIR)
os.environ['SCAX_DRIVE_ROOT'] = DRIVE_ROOT
sys.path.insert(0, f'{REPO_DIR}/src')
pull = subprocess.run(['git', 'pull', 'origin', 'main'], capture_output=True, text=True)
print(pull.stdout.strip() or pull.stderr.strip())

In [ ]:
import json, time
import numpy as np
import pandas as pd
from scax import paths
from scax.utils import seed_everything, write_json
from scax.data import public, splits
from scax.experiments.main_results import FrozenTask
from scax.experiments import development as dev
from scax.experiments import model_ablations as ma
from scax.experiments import runtime as rt
from scax.models import train
from scax.models.export import Ensemble
from sklearn.metrics import roc_auc_score
from scax.models.predictor import ExactPredictor

SEED = 42
seed_everything(SEED)
TREES, DEPTHS = [25, 50, 100, 200], {'frozen settings': [3, 4, 5, 6], 'no regularisation': [3, 4, 6]}
N_CELLS, TIME_BUDGET, CELL_CAP = 100, 240, 120     # distinct cells per model; seconds per model; seconds per cell
SERIES = {'frozen settings': {}, 'no regularisation': {'gamma': 0, 'min_child_weight': 1}}
TASKS = ['cdc_diabetes', 'brfss2015_heart', 'cardio70k']
SCALING_TASKS = ['brfss2015_heart', 'cardio70k']
TASK_NAMES = {'cdc_diabetes': 'CDC Diabetes', 'brfss2015_heart': 'BRFSS Heart disease', 'cardio70k': 'Cardio-70k'}
FAMILY_NAMES = {'xgboost': 'XGBoost', 'lightgbm': 'LightGBM', 'sklearn_rf': 'random forest'}
REC05, REC09C = json.load(open(f'{paths.RESULTS}/05_frozen.json')), json.load(open(f'{paths.RESULTS}/09c_frozen.json'))
FROZEN_SHA = {**REC05['frozen_config_sha256'], 'cardio70k': REC09C['frozen_config_sha256']['cardio70k']}
FT = {t: FrozenTask(t, REPO_DIR, DRIVE_ROOT, FROZEN_SHA[t]) for t in TASKS}
RUN_DIR = paths.DERIVED / 'runtime'; RUN_DIR.mkdir(parents=True, exist_ok=True)
HW = rt.hardware()
print('this session:', HW)
DATA = {}
for t in TASKS:
    X, y = public.LOADERS[t](paths.RAW)
    code = np.load(paths.DERIVED / 'splits' / f'{t}.npy')
    assert splits.summary(code)['sha256'] == FT[t].cfg['split_sha256']
    DATA[t] = {'Xa': X.to_numpy(dtype=float), 'y': y, 'code': code, 'names': list(X.columns)}
    del X

## Production runtime of the main results (from the stored candidate caches)

In [ ]:
PROD = []
for t in TASKS:
    ft, D = FT[t], DATA[t]
    rows = np.r_[ft.cal2_rows, ft.test_rows]
    units = [('xgboost', ft.pred, paths.DERIVED / 'certificates' / f'{t}__candidates.pkl', ft.ens)]
    for fam in ('lightgbm', 'sklearn_rf'):
        mf = paths.DERIVED / 'families' / f'{t}__{fam}__model.json'
        cf = paths.DERIVED / 'families' / f'{t}__{fam}__candidates.pkl'
        if mf.exists() and cf.exists():
            ens = Ensemble.load(mf); units.append((fam, ExactPredictor(ens), cf, ens))
    for fam, pred, cf, ens in units:
        C = pred.grid.cells(D['Xa'][rows])
        keys = [tuple(int(c[j]) for j in pred.grid.active) for c in C]
        r = rt.cache_runtime(cf, keys)
        PROD.append({'task': t, 'family': fam, 'explained_patients': len(rows), 'distinct_cells': len(set(keys)),
                     'cell_cache_hit_rate': 1 - len(set(keys)) / len(rows), **rt.model_size(ens, pred.grid), **r})
PROD = pd.DataFrame(PROD)
pd.set_option('display.width', 250)
print(PROD[['task', 'family', 'explained_patients', 'distinct_cells', 'cell_cache_hit_rate', 'trees', 'trees_with_a_split',
            'internal_nodes', 'distinct_thresholds', 'cells', 'median_seconds', 'q25_seconds', 'q75_seconds', 'p95_seconds',
            'max_seconds', 'memo_hit_rate', 'precheck_decided_share', 'search_unknown']].round(3).to_string(index=False))

## Timing study: frozen models and scaling with trees and depth

In [ ]:
def timing_cells(t, pred):
    """Cells of the explained test patients in a seeded random order (features only)."""
    rows = np.random.default_rng(SEED + 5).permutation(FT[t].test_rows)
    C = pred.grid.cells(DATA[t]['Xa'][rows])
    return C, pred.predict_cells(C).astype(int)


def run_model(t, series, n_trees, depth):
    """Fit (or reuse the frozen model), check it, time up to N_CELLS distinct cells; saved on Drive."""
    f = RUN_DIR / f"{t}__{series.replace(' ', '_')}__{n_trees}__{depth}.json"
    if f.exists():
        out = json.load(open(f))
        if (out['n_cells_requested'], out['time_budget'], out['cell_cap']) == (N_CELLS, TIME_BUDGET, CELL_CAP):
            return out
    ft, D = FT[t], DATA[t]
    params = {**ft.cfg['model']['params'], 'n_estimators': n_trees, 'max_depth': depth, **SERIES[series]}
    tr = splits.indices(D['code'], 'train')
    t0 = time.time()
    model, ens, pred = dev.fit_exported(D['Xa'][tr], D['y'][tr], params, SEED, D['names'], ft.ens.input_cast)
    fit_seconds = time.time() - t0
    oof_auroc = roc_auc_score(D['y'][tr], train.oof_scores('xgboost', D['Xa'][tr], D['y'][tr], params, SEED))
    frozen = params == ft.cfg['model']['params']
    if frozen:
        assert ens.hash() == ft.cfg['model']['sha256'], f'{t}: refitting does not reproduce the frozen model'
    agree = ma.exported_vs_native_detail(model, 'xgboost', pred, D['Xa'][tr])
    assert agree['beyond_rounding'] == 0, f'{t} {n_trees} trees depth {depth}: exported and native disagree {agree}'
    C, yh = timing_cells(t, pred)
    tg = rt.TimedGenerator(pred, M=ft.cfg['candidates']['M'], seed=ft.cfg['candidates']['order_seed'],
                           node_budget=ft.cfg['candidates']['node_budget'])
    t1 = time.time()
    rows = tg.time_cells(C, yh, time_budget=TIME_BUDGET, max_cells=N_CELLS, cell_cap=CELL_CAP)
    out = {'task': t, 'series': series, 'n_trees': n_trees, 'max_depth': depth, 'frozen_model': frozen, 'model_sha256': ens.hash(),
           'n_cells_requested': N_CELLS, 'time_budget': TIME_BUDGET, 'cell_cap': CELL_CAP, 'stopped_by_time': len(rows) < N_CELLS,
           'fit_seconds': fit_seconds, 'oof_auroc_training_portion': oof_auroc, 'timing_seconds': time.time() - t1, 'hardware': HW,
           'size': rt.model_size(ens, pred.grid), 'exported_vs_native': agree, 'rows': rows}
    json.dump(out, open(f, 'w'), default=float)
    return out


RUNS = []
plan = [(t, 'frozen settings', 100, 4) for t in TASKS if t not in SCALING_TASKS]
plan += [(t, sr, n, d) for sr in SERIES for t in SCALING_TASKS for d in DEPTHS[sr] for n in TREES]
for t, sr, n, d in plan:
    out = run_model(t, sr, n, d)
    s = rt.spread([r['seconds'] for r in out['rows']], [r['censored'] for r in out['rows']])
    print(f"{TASK_NAMES[t]}, {sr}: {n} trees, depth {d} -> {out['size']['trees_with_a_split']} trees with a split, "
          f"{out['size']['internal_nodes']} internal nodes, AUROC {out['oof_auroc_training_portion']:.4f}; {s['cells']} cells "
          f"({s['censored_cells']} censored), median {s['median_seconds']:.3f}s{' or more' if s['median_is_lower_bound'] else ''}"
          + (' (time limit reached)' if out['stopped_by_time'] else ''), flush=True)
    RUNS.append(out)

## Summary tables, RQ6 criterion and gate G5

In [ ]:
SCALE, CELLS = [], []
for out in RUNS:
    base = {'task': out['task'], 'series': out['series'], 'n_trees': out['n_trees'], 'max_depth': out['max_depth'],
            'frozen_model': out['frozen_model']}
    done = [r for r in out['rows'] if not r['censored']]
    SCALE.append({**base, **out['size'], 'oof_auroc_training_portion': out['oof_auroc_training_portion'],
                  **rt.spread([r['seconds'] for r in out['rows']], [r['censored'] for r in out['rows']]),
                  **{k: v for k, v in rt.phase_summary(done).items()},
                  'stopped_by_time': out['stopped_by_time'], 'fit_seconds': out['fit_seconds'], 'cpu': out['hardware']['cpu']})
    CELLS += [{**base, **r} for r in out['rows']]
SCALE, CELLS = pd.DataFrame(SCALE), pd.DataFrame(CELLS)
SCALE.to_csv(f'{paths.RESULTS}/15_runtime.csv', index=False)
CELLS.to_csv(f'{paths.RESULTS}/15_runtime_cells.csv', index=False)
PROD.to_csv(f'{paths.RESULTS}/15_runtime_production.csv', index=False)
print(SCALE[['task', 'series', 'n_trees', 'max_depth', 'trees_with_a_split', 'internal_nodes', 'distinct_thresholds',
             'oof_auroc_training_portion', 'cells', 'censored_cells', 'median_seconds', 'median_is_lower_bound', 'q25_seconds',
             'q75_seconds', 'p95_seconds', 'search_share', 'inflation_share', 'search_unknown', 'stopped_by_time']].round(3).to_string(index=False))
RQ6 = {'criterion': 'median of 5 seconds or less per distinct cell for 100 trees or fewer and depth 6 or less'}
for sr in SERIES:
    small = SCALE[(SCALE.series == sr) & (SCALE.n_trees <= 100) & (SCALE.max_depth <= 6)]
    fail = small[small.median_seconds > 5]
    RQ6[sr] = {'largest_median_seconds': float(small.median_seconds.max()), 'met': bool(fail.empty),
               'models_above_5_seconds': [f"{r.task} {r.n_trees} trees depth {r.max_depth}" for r in fail.itertuples()]}
G5 = {'rule': 'median above 30 seconds per cell for a frozen model',
      'production_median_seconds': {f"{r.task} {r.family}": float(r.median_seconds) for r in PROD.itertuples()},
      'passed': bool((PROD.median_seconds <= 30).all())}
print(json.dumps({'RQ6': RQ6, 'G5': G5}, indent=1))

## Figure: runtime

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.ticker import LogLocator, FuncFormatter, NullFormatter
plt.rcParams.update({'font.family': 'DejaVu Sans', 'font.size': 7.5, 'axes.spines.top': False, 'axes.spines.right': False})
DEPTH_COLOURS = dict(zip([3, 4, 5, 6], ['#9ecae1', '#4292c6', '#08519c', '#08306b']))
fig = plt.figure(figsize=(7.4, 5.4))
gs = fig.add_gridspec(2, 2, height_ratios=[1.15, 1], hspace=0.55, wspace=0.28, top=0.84)
secfmt = FuncFormatter(lambda v, _: f'{v:g} s')
for j, t in enumerate(SCALING_TASKS):
    ax = fig.add_subplot(gs[0, j])
    d = SCALE[SCALE.task == t]
    for sr, mk, filled in (('frozen settings', 'o', True), ('no regularisation', 's', False)):
        for depth in DEPTHS[sr]:
            e = d[(d.series == sr) & (d.max_depth == depth)].sort_values('n_trees')
            col = DEPTH_COLOURS[depth]
            ax.plot(e.internal_nodes, e.median_seconds, color=col, lw=1.1, ls='-' if filled else '--', zorder=2)
            ok, lb = e[~e.median_is_lower_bound], e[e.median_is_lower_bound]
            ax.errorbar(ok.internal_nodes, ok.median_seconds, yerr=[ok.median_seconds - ok.q25_seconds, ok.q75_seconds - ok.median_seconds],
                        fmt=mk, ms=3.8, color=col, mfc=col if filled else 'white', mec=col, mew=1.0, elinewidth=0.7, capsize=0, zorder=3)
            ax.plot(lb.internal_nodes, lb.median_seconds, '^', ms=5, color=col, mfc='white', mec=col, mew=1.0, zorder=3)
    f0 = d[d.frozen_model]
    ax.plot(f0.internal_nodes, f0.median_seconds, 'o', ms=10, mfc='none', mec='#D55E00', mew=1.4, zorder=4)
    ax.axhline(5, color='0.3', ls='--', lw=0.9)
    ax.axhline(CELL_CAP, color='0.3', ls=':', lw=0.9)
    ax.set_xscale('log'); ax.set_yscale('log'); ax.yaxis.set_major_formatter(secfmt)
    ax.xaxis.set_major_locator(LogLocator(base=10, subs=(1, 2, 5)))
    ax.xaxis.set_major_formatter(FuncFormatter(lambda v, _: f'{v:,.0f}'))
    ax.xaxis.set_minor_formatter(NullFormatter())
    ax.set_xlabel('Internal nodes in the ensemble (effective size)'); ax.set_title(TASK_NAMES[t], fontsize=8.5)
    ax.grid(color='0.92', lw=0.6); ax.set_axisbelow(True)
    if j == 0:
        ax.set_ylabel('Seconds per distinct cell (median, IQR)')
from matplotlib.lines import Line2D
h = [Line2D([], [], color=DEPTH_COLOURS[k], lw=1.4, label=f'Depth {k}') for k in sorted(DEPTH_COLOURS) if any(k in v for v in DEPTHS.values())]
h += [Line2D([], [], marker='o', color='0.3', ls='-', ms=4, label='Frozen settings (gamma 2, min. child weight 50)'),
      Line2D([], [], marker='s', color='0.3', mfc='white', ls='--', ms=4, label='No regularisation'),
      Line2D([], [], marker='^', color='0.3', mfc='white', ls='none', ms=5, label='Median is a lower bound (cells stopped)'),
      Line2D([], [], marker='o', color='#D55E00', mfc='none', ls='none', ms=8, label='Frozen model'),
      Line2D([], [], color='0.3', ls='--', lw=0.9, label='RQ6 criterion (5 s)'),
      Line2D([], [], color='0.3', ls=':', lw=0.9, label=f'Cell time cap ({CELL_CAP} s)')]
fig.legend(handles=h, loc='upper center', ncol=4, frameon=False, bbox_to_anchor=(0.5, 0.995), fontsize=7)
ax = fig.add_subplot(gs[1, :])
labels, segs = [], []
for t in TASKS:
    s = SCALE[(SCALE.task == t) & SCALE.frozen_model].iloc[0]
    for stage, name in (('deletion', 'deletion (AXps)'), ('inflation', 'inflation')):
        pre, sea, tot = s[f'mean_{stage}_precheck_seconds'], s[f'mean_{stage}_search_seconds'], s[f'mean_{stage}_seconds']
        labels.append(f'{TASK_NAMES[t]}, {name}'); segs.append((pre, sea, max(0.0, tot - pre - sea)))
segs = np.array(segs); yy = np.arange(len(labels))[::-1]
left = np.zeros(len(labels))
for k, (name, col) in enumerate([('Interval pre-check', '#56B4E9'), ('Exact search', '#0072B2'), ('Other (memo, bookkeeping)', '#BBBBBB')]):
    ax.barh(yy, segs[:, k], left=left, color=col, height=0.62, label=name, edgecolor='white', linewidth=0.6)
    left += segs[:, k]
ax.set_yticks(yy); ax.set_yticklabels(labels)
ax.set_xlabel('Mean seconds per distinct cell, frozen models, by step')
ax.xaxis.set_major_formatter(secfmt)
ax.legend(frameon=False, loc='lower right', fontsize=7)
ax.grid(axis='x', color='0.92', lw=0.6); ax.set_axisbelow(True)
for ext in ('png', 'pdf'):
    fig.savefig(paths.FIGURES / f'fig_runtime.{ext}', dpi=600, bbox_inches='tight')
plt.show()

In [ ]:
write_json(f'{paths.RESULTS}/15_runtime.json', {
    'frozen_config_sha256': FROZEN_SHA, 'hardware_this_session': HW, 'trees': TREES, 'depths': DEPTHS,
    'cells_per_model': N_CELLS, 'time_budget_seconds': TIME_BUDGET, 'cell_cap_seconds': CELL_CAP, 'series': SERIES, 'RQ6': RQ6, 'G5': G5,
    'production': PROD.to_dict('records'),
    'frozen_models': SCALE[SCALE.frozen_model].drop(columns=['cpu']).to_dict('records')})
print('written: results/15_runtime.csv, 15_runtime_cells.csv, 15_runtime_production.csv, 15_runtime.json; figures/fig_runtime')

In [ ]:
# === STANDARD COMMIT CELL ===
import os, subprocess
os.chdir(REPO_DIR)
r = subprocess.run(["python", "tools/commit_cell.py",
  "15: runtime: production seconds per cell of the main results, timing by step for the frozen models, scaling with 25 to 200 trees under the frozen settings and without regularisation on BRFSS Heart and Cardio-70k, effective model size, RQ6 and gate G5, figure"],
  capture_output=True, text=True)
print(r.stdout); print(r.stderr)